<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [148]:
import pandas as pd

df=pd.read_csv('/content/Titanic-Dataset.csv')
#print(df.shape)
#print(df.dtypes)
df.head()
#print(df.isnull().sum())  # drop Name,Cabin,PassengerId
#df.describe(include='all')
#df.duplicated().sum()
#df['Survived'].value_counts()


# drop Name,Cabin,PassengerId,Ticket
# need to fill the missing values of Age,Embarked

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [149]:
import seaborn as sns
import matplotlib.pyplot as plt
#sns.countplot(x='Survived', data=df)
#sns.countplot(x='Sex', hue='Survived', data=df)
#sns.countplot(x='Pclass', hue='Survived', data=df)
#sns.countplot(x='SibSp', hue='Survived', data=df)
#sns.countplot(x='Parch', hue='Survived', data=df)


In [150]:
df = df[['Name','Survived', 'Pclass', 'Sex', 'Age','SibSp','Parch','Fare','Embarked']]
for i, name in enumerate(df['Name']):
    if 'Mrs.' in name:
        df.loc[i, 'Name'] = 0
    elif 'Miss.' in name:
        df.loc[i, 'Name'] = 1
    elif 'Master.' in name:
        df.loc[i, 'Name'] = 2
    elif 'Mr.' in name:
        df.loc[i, 'Name'] = 3
    else:
        df.loc[i, 'Name'] = 4
df['Name'] = df['Name'].astype(int)
df.head()
print(df.nunique())
print(df.isnull().sum())
df.head()

Name          5
Survived      2
Pclass        3
Sex           2
Age          88
SibSp         7
Parch         7
Fare        248
Embarked      3
dtype: int64
Name          0
Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64


,Name,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,3,male,22.0,1,0,7.2500,S
1,0,1,1,female,38.0,1,0,71.2833,C
2,1,1,3,female,26.0,0,0,7.9250,S
3,0,1,1,female,35.0,1,0,53.1000,S
4,3,0,3,male,35.0,0,0,8.0500,S


In [151]:
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Sex']=df['Sex'].map({'male':0,'female':1})
df['Embarked']=df['Embarked'].map({'S':0,'C':1,'Q':2})
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df.head()

,Name,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone
0,3,0,3,0,22.0,1,0,7.2500,0,2,0
1,0,1,1,1,38.0,1,0,71.2833,1,2,0
2,1,1,3,1,26.0,0,0,7.9250,0,1,1
3,0,1,1,1,35.0,1,0,53.1000,0,2,0
4,3,0,3,0,35.0,0,0,8.0500,0,1,1


In [152]:
known_age=df[df['Age'].notnull()]
miss_age=df[df['Age'].isnull()]
features = ['Name','Pclass', 'Sex', 'Fare', 'SibSp', 'Fare','Embarked']

from sklearn.ensemble import RandomForestRegressor
model=RandomForestRegressor(n_estimators=100)
X=known_age[features]
y=known_age['Age']
model.fit(X,y)
pred=model.predict(miss_age[features])
print(pred)
df.loc[df['Age'].isnull(), 'Age'] = pred


[41.12143723 31.71484895 29.06666667 32.68227976 19.96146429 27.11640743
 34.52       21.99311905 24.79414863 32.37368783 30.47525121 39.11584794
 21.99311905 23.71333333 38.32508333 35.4405      6.465225   27.11640743
 30.47525121 20.58013095 30.47525121 30.47525121 27.11640743 29.57713567
  5.12579167 30.47525121 44.66946699  4.35570833 31.94       30.8347178
 25.05696219  9.3762619  40.17166667 52.5725      6.28757143 15.18454762
 29.38128205 46.87       30.1        44.66946699 21.99311905 16.39640476
 38.16663564 27.11640743  6.09555556 21.695      14.2934      6.08249167
 30.8347178  51.055      44.66946699 21.99311905 45.89733333 21.99311905
 35.72554442 55.48033333 35.4405     40.71733333 21.99311905 31.16293128
 25.45404625 30.47525121 29.58866667 16.39640476  4.91318452 35.18
 27.11640743 28.19       50.19083333 32.68227976 19.96146429 19.96146429
 39.11584794 29.06666667 21.99311905 37.15       27.11640743 24.023
  6.09555556 27.11640743 23.03971429 35.72554442 29.97666667 32

In [153]:
df.isnull().sum()
df.head()

,Name,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone
0,3,0,3,0,22.0,1,0,7.2500,0,2,0
1,0,1,1,1,38.0,1,0,71.2833,1,2,0
2,1,1,3,1,26.0,0,0,7.9250,0,1,1
3,0,1,1,1,35.0,1,0,53.1000,0,2,0
4,3,0,3,0,35.0,0,0,8.0500,0,1,1


In [157]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
print(df.dtypes)

X=df.drop('Survived', axis=1)
y=df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Name            int64
Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked        int64
FamilySize      int64
IsAlone         int64
dtype: object
Accuracy: 0.8603351955307262
